In [1]:
!pip install -q mcp anthropic groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.0/350.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.9 MB/s eta 0:00:00


In [2]:
import sqlite3
import json

# Create a banking database
conn = sqlite3.connect("banking.db")
cursor = conn.cursor()

# Create clients table
cursor.execute("""
CREATE TABLE IF NOT EXISTS clients (
    client_id TEXT PRIMARY KEY,
    name TEXT,
    balance FLOAT,
    transactions INTEGER,
    is_active BOOLEAN,
    sector TEXT,
    country TEXT,
    risk_level TEXT
)
""")

# Create transactions table
cursor.execute("""
CREATE TABLE IF NOT EXISTS transactions (
    transaction_id TEXT PRIMARY KEY,
    client_id TEXT,
    amount FLOAT,
    country TEXT,
    transaction_type TEXT,
    flagged BOOLEAN
)
""")

# Insert sample clients
clients = [
    ("NL-001", "Jan de Vries", 750.0, 142, True, "Manufacturing", "Netherlands", "HIGH"),
    ("NL-002", "Sara Ahmed", 12000.0, 45, True, "Real Estate", "Netherlands", "LOW"),
    ("NL-003", "Emma Koch", 3500.0, 89, True, "Energy", "Netherlands", "MEDIUM"),
    ("NL-004", "Peter Bakker", 450.0, 200, False, "Transport", "Netherlands", "HIGH"),
    ("NL-005", "Maria Santos", 25000.0, 30, True, "Finance", "Netherlands", "LOW"),
]

cursor.executemany("""
INSERT OR REPLACE INTO clients
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", clients)

# Insert sample transactions
transactions = [
    ("TX-001", "NL-001", 15000.0, "Country_A", "transfer", True),
    ("TX-002", "NL-002", 5000.0, "Netherlands", "payment", False),
    ("TX-003", "NL-003", 12000.0, "Country_B", "transfer", True),
    ("TX-004", "NL-004", 800.0, "Netherlands", "payment", False),
    ("TX-005", "NL-005", 50000.0, "Netherlands", "investment", False),
]

cursor.executemany("""
INSERT OR REPLACE INTO transactions
VALUES (?, ?, ?, ?, ?, ?)
""", transactions)

conn.commit()
conn.close()

print("✅ Banking database created!")
print("📊 Tables: clients, transactions")
print(f"👥 Clients: {len(clients)}")
print(f"💳 Transactions: {len(transactions)}")

✅ Banking database created!
📊 Tables: clients, transactions
👥 Clients: 5
💳 Transactions: 5


In [3]:
from mcp.server import Server
from mcp import types
from mcp.types import Tool, TextContent
import sqlite3
import json
import asyncio
from google.colab import userdata

# --------------------------------
# MCP Tool Logic
# --------------------------------
def execute_tool(name: str, arguments: dict):
    """Execute banking database tools"""
    conn = sqlite3.connect("banking.db")
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    try:
        if name == "get_client":
            client_id = arguments["client_id"]
            cursor.execute("SELECT * FROM clients WHERE client_id = ?", (client_id,))
            row = cursor.fetchone()
            result = dict(row) if row else {"error": f"Client {client_id} not found"}

        elif name == "get_high_risk_clients":
            cursor.execute("SELECT * FROM clients WHERE risk_level = 'HIGH'")
            result = [dict(row) for row in cursor.fetchall()]

        elif name == "get_flagged_transactions":
            cursor.execute("""
                SELECT t.*, c.name as client_name
                FROM transactions t
                JOIN clients c ON t.client_id = c.client_id
                WHERE t.flagged = 1
            """)
            result = [dict(row) for row in cursor.fetchall()]

        elif name == "search_clients":
            query = "SELECT * FROM clients WHERE 1=1"
            params = []
            if "sector" in arguments:
                query += " AND sector = ?"
                params.append(arguments["sector"])
            if "country" in arguments:
                query += " AND country = ?"
                params.append(arguments["country"])
            cursor.execute(query, params)
            result = [dict(row) for row in cursor.fetchall()]

        else:
            result = {"error": f"Unknown tool: {name}"}

    finally:
        conn.close()

    return json.dumps(result, indent=2)

# --------------------------------
# Available Tools Definition
# --------------------------------
TOOLS = [
    {
        "name": "get_client",
        "description": "Get client details from banking database by client ID",
        "parameters": {
            "type": "object",
            "properties": {
                "client_id": {"type": "string", "description": "Client ID e.g. NL-001"}
            },
            "required": ["client_id"]
        }
    },
    {
        "name": "get_high_risk_clients",
        "description": "Get all high risk clients from the banking database",
        "parameters": {"type": "object", "properties": {}}
    },
    {
        "name": "get_flagged_transactions",
        "description": "Get all flagged transactions requiring AML review",
        "parameters": {"type": "object", "properties": {}}
    },
    {
        "name": "search_clients",
        "description": "Search clients by sector or country",
        "parameters": {
            "type": "object",
            "properties": {
                "sector": {"type": "string", "description": "Sector e.g. Manufacturing"},
                "country": {"type": "string", "description": "Country e.g. Netherlands"}
            }
        }
    }
]

print("✅ MCP tool logic ready")
print(f"   {len(TOOLS)} tools defined")
for t in TOOLS:
    print(f"   → {t['name']}")

✅ MCP tool logic ready
   4 tools defined
   → get_client
   → get_high_risk_clients
   → get_flagged_transactions
   → search_clients


In [4]:
# Test all tools directly before connecting AI
print("🧪 Testing MCP Tools")
print("=" * 55)

print("\n📋 Test 1: Get client NL-001")
print(execute_tool("get_client", {"client_id": "NL-001"}))

print("\n🔴 Test 2: High risk clients")
print(execute_tool("get_high_risk_clients", {}))

print("\n🚨 Test 3: Flagged transactions")
print(execute_tool("get_flagged_transactions", {}))

print("\n🔍 Test 4: Energy sector clients")
print(execute_tool("search_clients", {"sector": "Energy"}))

🧪 Testing MCP Tools

📋 Test 1: Get client NL-001
{
  "client_id": "NL-001",
  "name": "Jan de Vries",
  "balance": 750.0,
  "transactions": 142,
  "is_active": 1,
  "sector": "Manufacturing",
  "country": "Netherlands",
  "risk_level": "HIGH"
}

🔴 Test 2: High risk clients
[
  {
    "client_id": "NL-001",
    "name": "Jan de Vries",
    "balance": 750.0,
    "transactions": 142,
    "is_active": 1,
    "sector": "Manufacturing",
    "country": "Netherlands",
    "risk_level": "HIGH"
  },
  {
    "client_id": "NL-004",
    "name": "Peter Bakker",
    "balance": 450.0,
    "transactions": 200,
    "is_active": 0,
    "sector": "Transport",
    "country": "Netherlands",
    "risk_level": "HIGH"
  }
]

🚨 Test 3: Flagged transactions
[
  {
    "transaction_id": "TX-001",
    "client_id": "NL-001",
    "amount": 15000.0,
    "country": "Country_A",
    "transaction_type": "transfer",
    "flagged": 1,
    "client_name": "Jan de Vries"
  },
  {
    "transaction_id": "TX-003",
    "client_id":

In [5]:
from groq import Groq

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def ask_ai_with_mcp(question):
    """AI queries real bank database via MCP tools"""
    print(f"\n❓ Question: {question}")
    print("-" * 55)

    # Convert tools to Groq format
    groq_tools = [{"type": "function", "function": t} for t in TOOLS]

    messages = [
        {
            "role": "system",
            "content": """You are a banking compliance officer.
            Use the available tools to query the bank database
            and answer questions accurately with real data."""
        },
        {"role": "user", "content": question}
    ]

    # First call — AI decides which tool to use
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=groq_tools,
        tool_choice="auto"
    )

    # Execute tool calls
    if response.choices[0].message.tool_calls:
        messages.append(response.choices[0].message)

        for tool_call in response.choices[0].message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)

            print(f"🔧 AI calling MCP tool: {tool_name}")
            print(f"   Args: {tool_args}")

            # Execute via MCP tool logic
            result = execute_tool(tool_name, tool_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

        # Final answer with tool results
        final = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )
        answer = final.choices[0].message.content
    else:
        answer = response.choices[0].message.content

    print(f"\n✅ ANSWER:\n{answer}")
    return answer

print("✅ AI + MCP integration ready")

✅ AI + MCP integration ready


In [6]:
questions = [
    "Who are all the high risk clients in our database?",
    "Show me details for client NL-003",
    "Which transactions are flagged for AML review?",
    "How many clients do we have in the Energy sector?"
]

print("🏦 AI QUERYING REAL BANK DATABASE VIA MCP")
print("=" * 55)

for question in questions:
    ask_ai_with_mcp(question)
    print()

🏦 AI QUERYING REAL BANK DATABASE VIA MCP

❓ Question: Who are all the high risk clients in our database?
-------------------------------------------------------
🔧 AI calling MCP tool: get_high_risk_clients
   Args: None

✅ ANSWER:
We have 2 high-risk clients in our database:

1. Jan de Vries (client ID: NL-001) with a balance of €750 and 142 transactions.
2. Peter Bakker (client ID: NL-004) with a balance of €450 and 200 transactions.

Please note that Peter Bakker's account is currently inactive.


❓ Question: Show me details for client NL-003
-------------------------------------------------------
🔧 AI calling MCP tool: get_client
   Args: {'client_id': 'NL-003'}

✅ ANSWER:
Here are the details for client NL-003:
- Client ID: NL-003
- Name: Emma Koch
- Balance: $3500.00
- Number of Transactions: 89
- Account Status: Active
- Sector: Energy
- Country: Netherlands
- Risk Level: Medium


❓ Question: Which transactions are flagged for AML review?
-----------------------------------------

In [7]:
print("🏦 Banking AI — Connected to Real Database via MCP")
print("Ask anything about clients or transactions")
print("Type 'exit' to quit\n")

while True:
    question = input("❓ Your question: ")
    if question.lower() == 'exit':
        print("👋 Goodbye!")
        break
    ask_ai_with_mcp(question)

🏦 Banking AI — Connected to Real Database via MCP
Ask anything about clients or transactions
Type 'exit' to quit

❓ Your question: exit
👋 Goodbye!
